# upstream_actor_low_risk_refresh_v1

## Goal

Check whether any retained upstream actor can provide a current-coverage low-drawdown base before opening another PPO branch.

## Governance

- Baselines: canonical `balanced_rule_residual_v22`; low-risk reference `risk_budget_usage_v1`; controls `balanced` and retained residual-family actors.
- Role: `diagnose`.
- Target window: strict OOS from local qlib `test` coverage plus saved actor walk-forward windows `wf_2020`, `wf_2022`, `wf_2023_2024` when actor artifacts exist.
- Accepted trade-off: prefer lower drawdown only if buy-hold-relative excess stays at least `+30pp`; reject low-drawdown paths that achieve risk only by missing the SOXL recovery.
- Changed layer: `model_selection`; no actor retraining, no shared code changes.
- External references: Qlib keeps this inside a reproducible ML/backtest harness; Deep Momentum Networks motivate separating trend estimation from position sizing; Bailey et al. PBO motivates walk-forward rather than selecting only the prettiest OOS row; drawdown-risk literature motivates treating path drawdown as an explicit risk criterion.
- Hypothesis: at least one retained actor, especially `risk_budget_usage_v1` or regularized residual actors, still has current-data coverage and can serve as a lower-DD upstream base.
- Acceptance criteria: strict OOS `relative_return >= 30pp` and `max_dd <= 15%`, plus non-missing walk-forward evidence where actor tags exist. A stricter anti-overfit screen requires every exported WF window to keep `max_dd <= 15%` and `relative_return >= -10pp`.
- Expected trade-off: conservative actors may lower MaxDD while sacrificing rebound capture.
- Failure mode: no actor meets the drawdown target, or a low-DD actor has missing profile/WF evidence and cannot be safely promoted.
- Comparison-table row key: `upstream_actor_low_risk_refresh_v1`.

## Historical Reflection

- Closest success: `v19reg_v22_budget_usage_bull_guard_v1` showed that actor-derived slow/fast composition can improve selected WF windows without retraining.
- Closest failure: `current_coverage_residual_guard_refresh_v1` found no current-coverage low-DD base by tightening path-level caps on high-DD families.
- Why this is not a duplicate: this notebook refreshes actor-backed daily paths directly from retained actors and profiles, recording missing profile/actor coverage as a model-selection constraint before any new training branch.

References:

- Qlib: https://github.com/microsoft/qlib
- Deep Momentum Networks: https://arxiv.org/abs/1904.04912
- Backtest overfitting / PBO: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2308659
- Drawdown measure in portfolio optimization: https://link.springer.com/article/10.1007/s11590-024-02124-x


In [1]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "scripts/export_profile_daily_backtest.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "scripts/export_profile_daily_backtest.py").exists():
    raise RuntimeError("could not locate repository root")
CARD = ROOT / "runtime/model_selection/card/upstream_actor_low_risk_refresh_v1"
EXPORT_DIR = CARD / "exports"
LOG_DIR = CARD / "logs"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from scripts.export_profile_daily_backtest import resolve_model_path
from src.soxl_quant.settings import load_profile

WINDOWS = {
    "strict_oos": ("", ""),
    "wf_2020": ("2020-01-02", "2020-12-31"),
    "wf_2022": ("2022-01-03", "2022-12-30"),
    "wf_2023_2024": ("2023-01-03", "2024-12-31"),
}

CANDIDATES = [
    {"profile": "balanced", "why": "long-lived control actor", "wf": True},
    {"profile": "balanced_rule_residual_v22", "why": "canonical runtime actor", "wf": True},
    {"profile": "balanced_rule_residual_v19_reg", "why": "regularized residual actor with bear-window history", "wf": True},
    {"profile": "balanced_rule_residual_v19_reg_light", "why": "lighter regularization ablation", "wf": True},
    {"profile": "balanced_rule_residual_mode_v1", "why": "slow mode actor from budget-usage family", "wf": True},
    {"profile": "balanced_rule_residual_imitation_v4", "why": "fast imitation actor from budget-usage family", "wf": True},
    {"profile": "risk_budget_usage_v1", "why": "native budget-usage low-risk actor", "wf": False},
    {"profile": "balanced_macro_v1", "why": "macro-feature control actor", "wf": False},
]

PROFILE_AUDIT_NAMES = [c["profile"] for c in CANDIDATES] + ["balanced_rule_residual_v19_small"]
print(f"card={CARD.relative_to(ROOT)}")
print(f"candidate_profiles={len(CANDIDATES)}")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


card=runtime/model_selection/card/upstream_actor_low_risk_refresh_v1
candidate_profiles=8


In [2]:
def actor_status(profile: str, tag: str = "") -> dict:
    row = {"profile": profile, "tag": tag or "strict_oos"}
    try:
        loaded = load_profile(profile)
        row["profile_ok"] = True
        row["position_mapping"] = getattr(loaded.model, "position_mapping", "")
    except Exception as exc:
        row.update({"profile_ok": False, "position_mapping": "", "error": f"{type(exc).__name__}: {exc}"})
        return row
    try:
        model_path = resolve_model_path(profile, tag)
        row.update({"actor_ok": True, "actor_path": str(model_path.relative_to(ROOT))})
    except Exception as exc:
        row.update({"actor_ok": False, "actor_path": "", "error": f"{type(exc).__name__}: {exc}"})
    return row

audit_rows = []
for profile in PROFILE_AUDIT_NAMES:
    audit_rows.append(actor_status(profile, ""))
    if any(c["profile"] == profile and c.get("wf") for c in CANDIDATES):
        for tag in ("wf_2020", "wf_2022", "wf_2023_2024"):
            audit_rows.append(actor_status(profile, tag))

profile_audit = pd.DataFrame(audit_rows)
profile_audit.to_csv(CARD / "profile_actor_audit.csv", index=False)
profile_audit


,profile,tag,profile_ok,position_mapping,actor_ok,actor_path,error
0,balanced,strict_oos,True,default,True,runtime/model/card/balanced/soxl_balanced_acto...,NaN
1,balanced,wf_2020,True,default,True,runtime/model/card/balanced_wf_2020/soxl_balan...,NaN
2,balanced,wf_2022,True,default,True,runtime/model/card/balanced_wf_2022/soxl_balan...,NaN
3,balanced,wf_2023_2024,True,default,True,runtime/model/card/balanced_wf_2023_2024/soxl_...,NaN
4,balanced_rule_residual_v22,strict_oos,True,default,True,runtime/model/card/balanced_rule_residual_v22/...,NaN
5,balanced_rule_residual_v22,wf_2020,True,default,True,runtime/model/card/balanced_rule_residual_v22_...,NaN
6,balanced_rule_residual_v22,wf_2022,True,default,True,runtime/model/card/balanced_rule_residual_v22_...,NaN
7,balanced_rule_residual_v22,wf_2023_2024,True,default,True,runtime/model/card/balanced_rule_residual_v22_...,NaN
8,balanced_rule_residual_v19_reg,strict_oos,True,default,True,runtime/model/card/balanced_rule_residual_v19_...,NaN
9,balanced_rule_residual_v19_reg,wf_2020,True,default,True,runtime/model/card/balanced_rule_residual_v19_...,NaN


In [3]:
def export_daily(profile: str, tag: str, start: str = "", end: str = "") -> dict:
    window = tag or "strict_oos"
    out = EXPORT_DIR / f"{profile}__{window}.csv"
    stdout_log = LOG_DIR / f"{profile}__{window}.stdout.log"
    stderr_log = LOG_DIR / f"{profile}__{window}.stderr.log"
    cmd = [
        sys.executable,
        "scripts/export_profile_daily_backtest.py",
        "--profile",
        profile,
        "--output",
        str(out),
    ]
    if tag:
        cmd.extend(["--tag", tag])
    if start and end:
        cmd.extend(["--test-start", start, "--test-end", end])
    env = dict(os.environ)
    env["PYTHONPATH"] = "src:."
    started = datetime.now(timezone.utc).isoformat()
    try:
        proc = subprocess.run(
            cmd,
            cwd=ROOT,
            env=env,
            capture_output=True,
            text=True,
            timeout=300,
        )
        stdout_log.write_text(proc.stdout, encoding="utf-8")
        stderr_log.write_text(proc.stderr, encoding="utf-8")
        ok = proc.returncode == 0 and out.exists()
        return {
            "profile": profile,
            "window": window,
            "tag": tag,
            "start": start,
            "end": end,
            "ok": ok,
            "returncode": proc.returncode,
            "csv": str(out.relative_to(ROOT)) if out.exists() else "",
            "stdout_log": str(stdout_log.relative_to(ROOT)),
            "stderr_log": str(stderr_log.relative_to(ROOT)),
            "started_at": started,
            "finished_at": datetime.now(timezone.utc).isoformat(),
            "error_tail": proc.stderr[-1200:] if proc.returncode else "",
        }
    except Exception as exc:
        stdout_log.write_text("", encoding="utf-8")
        stderr_log.write_text(f"{type(exc).__name__}: {exc}\n", encoding="utf-8")
        return {
            "profile": profile,
            "window": window,
            "tag": tag,
            "start": start,
            "end": end,
            "ok": False,
            "returncode": -1,
            "csv": "",
            "stdout_log": str(stdout_log.relative_to(ROOT)),
            "stderr_log": str(stderr_log.relative_to(ROOT)),
            "started_at": started,
            "finished_at": datetime.now(timezone.utc).isoformat(),
            "error_tail": f"{type(exc).__name__}: {exc}",
        }

export_jobs = []
for candidate in CANDIDATES:
    profile = candidate["profile"]
    export_jobs.append((profile, "", "", ""))
    if candidate.get("wf"):
        for tag in ("wf_2020", "wf_2022", "wf_2023_2024"):
            start, end = WINDOWS[tag]
            export_jobs.append((profile, tag, start, end))

export_rows = []
for idx, (profile, tag, start, end) in enumerate(export_jobs, start=1):
    row = export_daily(profile, tag, start, end)
    export_rows.append(row)
    print(f"{idx:02d}/{len(export_jobs)} {profile} {tag or 'strict_oos'} ok={row['ok']}")

export_status = pd.DataFrame(export_rows)
export_status.to_csv(CARD / "export_status.csv", index=False)
export_status[["profile", "window", "ok", "csv", "returncode"]]


01/26 balanced strict_oos ok=True


02/26 balanced wf_2020 ok=True


03/26 balanced wf_2022 ok=True


04/26 balanced wf_2023_2024 ok=True


05/26 balanced_rule_residual_v22 strict_oos ok=True


...

23/26 balanced_rule_residual_imitation_v4 wf_2022 ok=True


24/26 balanced_rule_residual_imitation_v4 wf_2023_2024 ok=True


25/26 risk_budget_usage_v1 strict_oos ok=True


26/26 balanced_macro_v1 strict_oos ok=True


,profile,window,ok,csv,returncode
0,balanced,strict_oos,True,runtime/model_selection/card/upstream_actor_lo...,0
1,balanced,wf_2020,True,runtime/model_selection/card/upstream_actor_lo...,0
2,balanced,wf_2022,True,runtime/model_selection/card/upstream_actor_lo...,0
3,balanced,wf_2023_2024,True,runtime/model_selection/card/upstream_actor_lo...,0
4,balanced_rule_residual_v22,strict_oos,True,runtime/model_selection/card/upstream_actor_lo...,0
5,balanced_rule_residual_v22,wf_2020,True,runtime/model_selection/card/upstream_actor_lo...,0
6,balanced_rule_residual_v22,wf_2022,True,runtime/model_selection/card/upstream_actor_lo...,0
7,balanced_rule_residual_v22,wf_2023_2024,True,runtime/model_selection/card/upstream_actor_lo...,0
8,balanced_rule_residual_v19_reg,strict_oos,True,runtime/model_selection/card/upstream_actor_lo...,0
9,balanced_rule_residual_v19_reg,wf_2020,True,runtime/model_selection/card/upstream_actor_lo...,0


In [4]:
def max_drawdown(nav: pd.Series) -> float:
    nav = nav.astype(float)
    return float((1.0 - nav / nav.cummax()).max() * 100.0)

def annualized_sharpe(nav: pd.Series) -> float:
    returns = nav.astype(float).pct_change().replace([np.inf, -np.inf], np.nan).dropna()
    if returns.empty or returns.std(ddof=0) == 0 or math.isnan(returns.std(ddof=0)):
        return float("nan")
    return float(np.sqrt(252.0) * returns.mean() / returns.std(ddof=0))

def metric_row(export_row: pd.Series) -> dict:
    base = export_row.to_dict()
    if not export_row.get("ok") or not export_row.get("csv"):
        return {**base, "metric_ok": False}
    path = ROOT / str(export_row["csv"])
    df = pd.read_csv(path, parse_dates=["datetime"]).sort_values("datetime")
    if df.empty:
        return {**base, "metric_ok": False, "metric_error": "empty_csv"}
    nav = df["strategy_nav"]
    bh = df["buyhold_nav"]
    strategy_return = float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)
    buyhold_return = float((bh.iloc[-1] / bh.iloc[0] - 1.0) * 100.0)
    action = df.get("action", pd.Series(["HOLD"] * len(df)))
    return {
        **base,
        "metric_ok": True,
        "rows": int(len(df)),
        "first_date": df["datetime"].iloc[0].date().isoformat(),
        "last_date": df["datetime"].iloc[-1].date().isoformat(),
        "strategy_return": strategy_return,
        "buyhold_return": buyhold_return,
        "relative_return": strategy_return - buyhold_return,
        "sharpe": annualized_sharpe(nav),
        "max_dd": max_drawdown(nav),
        "avg_pos": float(df["position"].astype(float).mean()),
        "trades": int((action != "HOLD").sum()),
    }

metrics = pd.DataFrame([metric_row(row) for _, row in export_status.iterrows()])
metrics["strict_target_pass"] = (
    metrics["metric_ok"].fillna(False)
    & (metrics["window"] == "strict_oos")
    & (metrics["relative_return"] >= 30.0)
    & (metrics["max_dd"] <= 15.0)
)
metrics["risk_target_pass"] = metrics["metric_ok"].fillna(False) & (metrics["max_dd"] <= 15.0)
metrics.to_csv(CARD / "actor_metrics.csv", index=False)
metrics.sort_values(["window", "max_dd", "relative_return"], ascending=[True, True, False])[
    ["profile", "window", "first_date", "last_date", "strategy_return", "buyhold_return", "relative_return", "sharpe", "max_dd", "avg_pos", "trades", "strict_target_pass"]
]


,profile,window,first_date,last_date,strategy_return,buyhold_return,relative_return,sharpe,max_dd,avg_pos,trades,strict_target_pass
24,risk_budget_usage_v1,strict_oos,2020-01-02,2026-05-21,1014.010398,816.512500,197.497898,1.059633,53.504991,0.437616,251,False
8,balanced_rule_residual_v19_reg,strict_oos,2020-01-02,2026-05-21,869.699640,816.512500,53.187140,1.003782,56.161498,0.449886,186,False
0,balanced,strict_oos,2020-01-02,2026-05-21,1172.222473,816.512500,355.709973,1.025160,56.239583,0.494162,338,False
4,balanced_rule_residual_v22,strict_oos,2020-01-02,2026-05-21,1036.112215,816.512500,219.599715,0.999409,56.922809,0.500889,172,False
16,balanced_rule_residual_mode_v1,strict_oos,2020-01-02,2026-05-21,804.987864,816.512500,-11.524635,0.954128,56.945387,0.474314,179,False
20,balanced_rule_residual_imitation_v4,strict_oos,2020-01-02,2026-05-21,1017.926497,816.512500,201.413997,1.022860,57.279834,0.475650,184,False
25,balanced_macro_v1,strict_oos,2020-01-02,2026-05-21,2184.314061,816.512500,1367.801561,1.110258,59.030511,0.585376,332,False
12,balanced_rule_residual_v19_reg_light,strict_oos,2020-01-02,2026-05-21,1075.665026,816.512500,259.152526,0.995131,62.354532,0.522875,174,False
9,balanced_rule_residual_v19_reg,wf_2020,2020-01-02,2020-12-30,112.269617,58.093571,54.176047,2.092273,21.143984,0.387232,30,False
13,balanced_rule_residual_v19_reg_light,wf_2020,2020-01-02,2020-12-30,112.269617,58.093571,54.176047,2.092273,21.143984,0.387232,30,False


In [5]:
strict = metrics[(metrics["window"] == "strict_oos") & metrics["metric_ok"].fillna(False)].copy()
strict_ranked = strict.sort_values(["strict_target_pass", "max_dd", "relative_return"], ascending=[False, True, False])

wf = metrics[(metrics["window"].isin(["wf_2020", "wf_2022", "wf_2023_2024"])) & metrics["metric_ok"].fillna(False)].copy()
wf_summary_rows = []
for profile, group in wf.groupby("profile"):
    windows = set(group["window"])
    wf_summary_rows.append({
        "profile": profile,
        "wf_window_count": len(windows),
        "has_all_wf": windows == {"wf_2020", "wf_2022", "wf_2023_2024"},
        "wf_min_relative": float(group["relative_return"].min()),
        "wf_max_dd": float(group["max_dd"].max()),
        "wf_mean_relative": float(group["relative_return"].mean()),
        "wf_mean_return": float(group["strategy_return"].mean()),
    })
wf_summary = pd.DataFrame(wf_summary_rows)
if not wf_summary.empty:
    wf_summary["anti_overfit_pass"] = (
        wf_summary["has_all_wf"]
        & (wf_summary["wf_max_dd"] <= 15.0)
        & (wf_summary["wf_min_relative"] >= -10.0)
    )

family = strict.merge(wf_summary, on="profile", how="left")
for col in ["wf_window_count", "wf_min_relative", "wf_max_dd", "wf_mean_relative", "wf_mean_return"]:
    if col in family.columns:
        family[col] = family[col].fillna(np.nan)
family["anti_overfit_pass"] = family.get("anti_overfit_pass", False).fillna(False)
family["full_pass"] = family["strict_target_pass"] & family["anti_overfit_pass"]
family = family.sort_values(["full_pass", "strict_target_pass", "max_dd", "relative_return"], ascending=[False, False, True, False])
family.to_csv(CARD / "actor_family_summary.csv", index=False)
family[["profile", "strategy_return", "buyhold_return", "relative_return", "sharpe", "max_dd", "avg_pos", "strict_target_pass", "has_all_wf", "wf_min_relative", "wf_max_dd", "anti_overfit_pass", "full_pass"]]


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_31276/2843381311.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  family["anti_overfit_pass"] = family.get("anti_overfit_pass", False).fillna(False)


,profile,strategy_return,buyhold_return,relative_return,sharpe,max_dd,avg_pos,strict_target_pass,has_all_wf,wf_min_relative,wf_max_dd,anti_overfit_pass,full_pass
6,risk_budget_usage_v1,1014.010398,816.5125,197.497898,1.059633,53.504991,0.437616,False,NaN,NaN,NaN,False,False
2,balanced_rule_residual_v19_reg,869.699640,816.5125,53.187140,1.003782,56.161498,0.449886,False,True,-161.595380,42.227591,False,False
0,balanced,1172.222473,816.5125,355.709973,1.025160,56.239583,0.494162,False,True,-132.187862,38.259409,False,False
1,balanced_rule_residual_v22,1036.112215,816.5125,219.599715,0.999409,56.922809,0.500889,False,True,-96.858160,34.845776,False,False
4,balanced_rule_residual_mode_v1,804.987864,816.5125,-11.524635,0.954128,56.945387,0.474314,False,True,-154.022195,41.292195,False,False
5,balanced_rule_residual_imitation_v4,1017.926497,816.5125,201.413997,1.022860,57.279834,0.475650,False,True,-154.929564,37.962106,False,False
7,balanced_macro_v1,2184.314061,816.5125,1367.801561,1.110258,59.030511,0.585376,False,NaN,NaN,NaN,False,False
3,balanced_rule_residual_v19_reg_light,1075.665026,816.5125,259.152526,0.995131,62.354532,0.522875,False,True,-124.551029,39.505500,False,False


In [6]:
strict_pass_count = int(family["strict_target_pass"].sum()) if not family.empty else 0
full_pass_count = int(family["full_pass"].sum()) if not family.empty else 0
best_low_dd = family.iloc[0].to_dict() if not family.empty else {}
best_excess = strict.sort_values("relative_return", ascending=False).iloc[0].to_dict() if not strict.empty else {}
best_risk = strict.sort_values("max_dd", ascending=True).iloc[0].to_dict() if not strict.empty else {}

decision = {
    "experiment": "upstream_actor_low_risk_refresh_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "strict_pass_count": strict_pass_count,
    "full_pass_count": full_pass_count,
    "decision": "diagnostic_no_current_actor_pass" if full_pass_count == 0 else "candidate_found_requires_review",
    "best_ranked_profile": best_low_dd.get("profile"),
    "best_ranked_relative_return": best_low_dd.get("relative_return"),
    "best_ranked_max_dd": best_low_dd.get("max_dd"),
    "best_excess_profile": best_excess.get("profile"),
    "best_excess_relative_return": best_excess.get("relative_return"),
    "best_excess_max_dd": best_excess.get("max_dd"),
    "lowest_dd_profile": best_risk.get("profile"),
    "lowest_dd_relative_return": best_risk.get("relative_return"),
    "lowest_dd_max_dd": best_risk.get("max_dd"),
    "next_action": "Do not promote a refreshed upstream actor as-is. If no strict/full pass exists, the next valid loop is a training-time or composition-level risk-state selector that explicitly optimizes drawdown without relying on path-level cap tightening.",
    "artifacts": {
        "profile_actor_audit": str((CARD / "profile_actor_audit.csv").relative_to(ROOT)),
        "export_status": str((CARD / "export_status.csv").relative_to(ROOT)),
        "actor_metrics": str((CARD / "actor_metrics.csv").relative_to(ROOT)),
        "actor_family_summary": str((CARD / "actor_family_summary.csv").relative_to(ROOT)),
    },
}
(CARD / "selection.json").write_text(json.dumps(decision, indent=2, sort_keys=True), encoding="utf-8")
decision


{'experiment': 'upstream_actor_low_risk_refresh_v1',
 'created_at': '2026-05-24T13:42:30.196688+00:00',
 'strict_pass_count': 0,
 'full_pass_count': 0,
 'decision': 'diagnostic_no_current_actor_pass',
 'best_ranked_profile': 'risk_budget_usage_v1',
 'best_ranked_relative_return': 197.4978984629156,
 'best_ranked_max_dd': 53.50499059490929,
 'best_excess_profile': 'balanced_macro_v1',
 'best_excess_relative_return': 1367.8015608391452,
 'best_excess_max_dd': 59.0305111830142,
 'lowest_dd_profile': 'risk_budget_usage_v1',
 'lowest_dd_relative_return': 197.4978984629156,
 'lowest_dd_max_dd': 53.50499059490929,
 'next_action': 'Do not promote a refreshed upstream actor as-is. If no strict/full pass exists, the next valid loop is a training-time or composition-level risk-state selector that explicitly optimizes drawdown without relying on path-level cap tightening.',
 'artifacts': {'profile_actor_audit': 'runtime/model_selection/card/upstream_actor_low_risk_refresh_v1/profile_actor_audit.cs

## Single-Run Conclusion

Decision: `diagnostic_no_current_actor_pass`.

- `strict_pass_count=0`, `full_pass_count=0`.
- Lowest strict-OOS MaxDD among refreshed actors is still `53.50%` from `risk_budget_usage_v1`; it has `+197.50pp` buy-hold-relative excess but fails the `<=15%` drawdown target by a wide margin.
- Highest strict-OOS excess is `balanced_macro_v1` at `+1367.80pp`, but MaxDD is `59.03%`, so it is not a low-risk candidate.
- All profiles with saved WF actors exported cleanly, but every WF family fails the anti-overfit drawdown screen: exported WF max drawdown is at least `34.85%`, far above `15%`.
- `balanced_rule_residual_v19_small` has a retained actor file but no loadable profile, so it is not reproducible through the repo-owned profile interface.

Conclusion: do not promote a refreshed upstream actor directly. The next valid loop should be training-time or composition-level risk-state selection that explicitly targets drawdown while preserving rebound participation, not another global path-level cap tightening pass.
